# 03 · CrawlSpider rules and feed exports

Crawl books.toscrape.com with link-extraction rules and export the results to CSV and JSON Lines.

*Crawl books.toscrape.com dengan aturan link extractor lalu ekspor ke CSV dan JSON Lines.*

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
// Packages / Paket
#r "nuget: Gravicode.ScrapyNet, 0.1.0"

// Before the packages are on nuget.org, point at a local build instead:
//   dotnet pack ScrapyNet.slnx -c Release -o artifacts
// and add this line above the #r lines:  #i "nuget: ../artifacts"

In [ ]:
using ScrapyNet;

public sealed class BooksSpider : CrawlSpider
{
    public override string Name => "books";
    public override IReadOnlyList<string> AllowedDomains => ["books.toscrape.com"];
    public override IReadOnlyList<string> StartUrls => ["https://books.toscrape.com/"];

    protected override IReadOnlyList<Rule> Rules =>
    [
        new(new LinkExtractor { RestrictCss = ["article.product_pod h3"] }, ParseBook),
    ];

    private IEnumerable<object> ParseBook(Response r)
    {
        yield return new
        {
            Title = r.Css("div.product_main h1::text").Get(),
            Price = ScrapyNet.Loaders.Transforms.Price(r.Css("p.price_color::text").Get()),
            Stock = ScrapyNet.Loaders.Transforms.ToInt(r.Css("p.availability").GetText()),
            Upc = r.Xpath("//th[text()='UPC']/following-sibling::td/text()").Get(),
        };
    }
}

In [ ]:
using ScrapyNet;

var output = Path.Combine(Path.GetTempPath(), "scrapynet-books");
Directory.CreateDirectory(output);
var settings = new Settings();
settings.Set(SettingKeys.LogLevel, "WARNING");
settings.Set(SettingKeys.CloseSpiderItemCount, 10);   // a small sample of the catalogue
settings.Set(SettingKeys.DownloadDelay, 0.25);
settings.Feeds.Add(Path.Combine(output, "books.csv"), new FeedOptions { Overwrite = true });
settings.Feeds.Add(Path.Combine(output, "books.jsonl"), new FeedOptions { Overwrite = true });

var result = await Scrapy.CrawlAsync(new BooksSpider(), settings);
display(result);
display(File.ReadLines(Path.Combine(output, "books.csv")).Take(4));